# Writing Tools NLP — QLoRA fine-tuning (Colab T4)

Fine-tune **Qwen3-4B-Instruct-2507** with QLoRA on ~4k writing-task examples.

**Runtime:** GPU → T4 (free tier). Do not run on Mac — `bitsandbytes` is CUDA-only.

**Steps:** clone repo → prepare data → train → download `adapter/` folder.

**Note:** the repo's `train/train.py` includes compatibility for current TRL versions, so you do not need manual Colab patch cells for `max_length` or `assistant_only_loss`.

In [ ]:
# Cell 1 — GPU check + clone repo
import torch

assert torch.cuda.is_available(), "Enable GPU: Runtime → Change runtime type → T4"
print(f"GPU: {torch.cuda.get_device_name(0)}")

# Clone (replace with your fork URL if needed)
!git clone https://github.com/Aditya-ice/writing-tools-NLP.git
%cd writing-tools-NLP

In [ ]:
# Cell 2 — Install dependencies (CUDA build of torch is preinstalled on Colab)
%pip install -q -r requirements.txt

# mlx is Apple-Silicon-only; if present on Colab its broken Linux wheel crashes
# transformers mid-training (libmlx.so ImportError). Make sure it is gone.
%pip uninstall -y -q mlx mlx-lm

In [ ]:
# Cell 3 — Prepare datasets (~4k examples, ~30s)
!python data/prepare.py

In [ ]:
# Cell 4 — Train QLoRA adapter (~1.5–2.5 h on T4 for 1 epoch)
# Resume after a Colab disconnect:
#   !python train/train.py --resume_from_checkpoint ./checkpoints/checkpoint-XXX

!python train/train.py

In [ ]:
# Cell 5 — Download adapter weights
from google.colab import files
import shutil

shutil.make_archive("adapter", "zip", "adapter")
files.download("adapter.zip")